# FSMW-DDS: Federated Self-Attention Wavelet Transformer with Diffusion-Based Data Synthesis
## Privacy-Preserving Cardiac Abnormality Detection from Phonocardiogram (PCG) Signals

This notebook is a **full, corrected, end-to-end implementation** of the framework described in the paper:

> *A Federated Self-Attention Wavelet Transformer with Diffusion-Based Data Synthesis for Privacy-Preserving Cardiac Abnormality Detection*

Pipeline implemented:

1. **Data loading** — PhysioNet/CinC Challenge 2016 Heart Sound dataset (with a synthetic-data fallback so the notebook runs end-to-end even without the dataset present).
2. **VMD** (Variational Mode Decomposition) — adaptive signal enhancement (Eqs. 2–7).
3. **MRWA** (Multi-Resolution Wavelet Analysis) — hierarchical time-frequency feature extraction (Eqs. 8–13).
4. **CDPM** (Conditional Diffusion Probabilistic Model) — synthetic minority-class (abnormal) sample generation (Eqs. 14–23).
5. **MSAWT** (Multi-Head Self-Attention Wavelet Transformer) — global + local cardiac pattern learning (Eqs. 24–37).
6. **FedProx** — federated, privacy-preserving optimization across simulated hospital clients (Eqs. 38–47).
7. **ACACC** — Adaptive Confidence-Aware Cardiac Abnormality Classification head (Eqs. 48–57).
8. **Evaluation** — the full metric suite used in the paper (accuracy, precision, recall, specificity, F1, MCC, NPV, balanced accuracy, Cohen's Kappa, G-Mean, Jaccard, Youden's Index, FPR, FNR, DOR, AUPRC), ablation study, 5-fold CV, noise-robustness analysis, and computational-cost comparison.

> **Note on data.** The original PhysioNet CinC 2016 dataset is fairly large and requires a Kaggle/PhysioNet download. This notebook looks for it at `DATA_DIR` (see Configuration cell) and, if not found, **automatically generates a physiologically-plausible synthetic PCG corpus** so every cell below is runnable top-to-bottom out of the box. Point `DATA_DIR` at your real download to reproduce paper-scale results.


## 0. Environment Setup
Installs the packages this notebook needs (skip cells already satisfied in your environment).

In [ ]:
# If a package is missing in your environment, uncomment the relevant line.
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# %pip install PyWavelets scikit-learn pandas numpy scipy matplotlib torchmetrics tqdm
print("If any import below fails, run the pip install cell above for the missing package.")


In [ ]:
import os
import math
import json
import random
import warnings
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import scipy.io.wavfile as wavfile
import scipy.signal as sps
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, cohen_kappa_score,
    balanced_accuracy_score, jaccard_score, roc_auc_score,
    average_precision_score
)
from scipy import stats

try:
    import pywt
    HAVE_PYWT = True
except ImportError:
    HAVE_PYWT = False
    warnings.warn("PyWavelets not found -- install with `pip install PyWavelets`. "
                   "A DWT fallback using numpy will NOT be used automatically; "
                   "please install pywt for full MRWA functionality.")

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 1. Configuration

All hyperparameters from the paper's Experimental Setup (Section 5.1) are centralised here.

In [ ]:
@dataclass
class Config:
    # ---- Data ----
    data_dir: str = "./physionet2016"        # expects wav files + REFERENCE.csv per training-a..e folders
    sample_rate: int = 2000                  # PCG recordings are resampled to 2 kHz
    segment_seconds: float = 5.0             # fixed-length segments extracted from each recording
    n_classes: int = 5                       # Normal, Murmur, Extrasystole, Artifact, Other Abnormalities
    class_names: tuple = ("Normal", "Murmur", "Extrasystole", "Artifact", "Other")

    # ---- VMD ----
    vmd_K: int = 5                           # number of intrinsic mode functions
    vmd_alpha: float = 2000.0                # bandwidth constraint (penalty factor)
    vmd_tau: float = 0.0                     # noise-tolerance (0 = strict fidelity)
    vmd_tol: float = 1e-7
    vmd_max_iter: int = 300

    # ---- MRWA ----
    wavelet: str = "db4"
    wp_level: int = 4                        # wavelet packet decomposition depth -> 2^4 = 16 sub-bands

    # ---- CDPM (diffusion) ----
    diffusion_timesteps: int = 200
    diffusion_hidden: int = 256
    diffusion_lr: float = 1e-3
    diffusion_epochs: int = 60
    diffusion_lambda_con: float = 0.1        # weight for feature-consistency loss (Eq. 22)

    # ---- MSAWT (transformer) ----
    d_model: int = 128
    n_heads: int = 8
    n_layers: int = 4
    ff_hidden: int = 256
    dropout: float = 0.1

    # ---- ACACC ----
    l2_reg: float = 1e-4                     # lambda in Eq. 54

    # ---- Federated learning (FedProx) ----
    n_clients: int = 5                       # simulated healthcare institutions
    fedprox_mu: float = 0.01                 # proximal regularization coefficient (Eq. 40)
    comm_rounds: int = 50
    local_epochs: int = 2
    client_lr: float = 1e-4

    # ---- Training ----
    batch_size: int = 32
    epochs: int = 50                         # used for the non-federated / ablation runs
    lr: float = 1e-4
    weight_decay: float = 1e-5

CFG = Config()
CFG


## 2. Data Loading — PhysioNet/CinC Challenge 2016

`load_physionet2016()` expects the standard Kaggle/PhysioNet layout:

```
data_dir/
  training-a/
    a0001.wav
    a0002.wav
    ...
    REFERENCE.csv        # columns: filename,label   (label: -1 normal, 1 abnormal)
  training-b/ ...
  ...
```

Since the 4-class breakdown (Murmur / Extrasystole / Artifact / Other) is only available if you also merge in the auxiliary annotation files that PhysioNet distributes separately, `load_physionet2016` falls back to the 2-class normal/abnormal labels found in `REFERENCE.csv` and upsamples them into the 5-class label space using the `label_map` you supply (edit this if your local copy already has fine-grained labels in a `label` column).

If `data_dir` does not exist, a **synthetic PCG corpus** is generated instead (Section 2.2) so the rest of the notebook is always runnable.

In [ ]:
def _synthesize_heart_sound(duration_s: float, fs: int, abnormal_class: int, rng: np.random.Generator) -> np.ndarray:
    """Physiologically-inspired synthetic PCG generator.

    class 0: Normal      -> clean S1/S2 pattern
    class 1: Murmur       -> S1/S2 + systolic murmur (broadband noise burst)
    class 2: Extrasystole -> S1/S2 + an extra irregular beat
    class 3: Artifact     -> heavy broadband noise, weak/absent heart sounds
    class 4: Other        -> S1/S2 with irregular timing / split S2
    """
    t = np.arange(0, duration_s, 1.0 / fs)
    sig = np.zeros_like(t)
    hr = rng.uniform(60, 100)                 # beats per minute
    beat_period = 60.0 / hr
    n_beats = int(duration_s / beat_period)

    def gauss_pulse(center, width, amp, freq):
        env = amp * np.exp(-0.5 * ((t - center) / width) ** 2)
        return env * np.sin(2 * np.pi * freq * (t - center))

    for b in range(n_beats):
        beat_start = b * beat_period + rng.normal(0, 0.01 if abnormal_class != 2 else 0.05)
        s1_t = beat_start + rng.uniform(0.0, 0.02)
        s2_t = beat_start + beat_period * rng.uniform(0.28, 0.35)

        sig += gauss_pulse(s1_t, 0.02, 1.0, 60)
        if abnormal_class == 4 and rng.random() < 0.5:
            sig += gauss_pulse(s2_t, 0.015, 0.6, 90)
            sig += gauss_pulse(s2_t + 0.02, 0.015, 0.5, 110)   # split S2
        else:
            sig += gauss_pulse(s2_t, 0.02, 0.8, 90)

        if abnormal_class == 1:  # murmur: systolic broadband burst between S1 and S2
            murmur_len = int((s2_t - s1_t) * fs * 0.7)
            start_idx = int(s1_t * fs)
            if murmur_len > 0 and start_idx + murmur_len < len(sig):
                murmur = rng.normal(0, 0.25, murmur_len) * np.hanning(murmur_len)
                sig[start_idx:start_idx + murmur_len] += murmur

        if abnormal_class == 2 and rng.random() < 0.3:  # extra irregular beat
            extra_t = beat_start + beat_period * rng.uniform(0.55, 0.7)
            sig += gauss_pulse(extra_t, 0.015, 0.7, 70)

    noise_level = 0.35 if abnormal_class == 3 else 0.03
    sig = sig + rng.normal(0, noise_level, len(sig))

    if abnormal_class == 3:  # artifact: attenuate true heart sounds, dominate with motion noise
        sig = 0.3 * sig + rng.normal(0, 0.4, len(sig))
        # low-frequency drift (motion artifact)
        drift = 0.3 * np.sin(2 * np.pi * rng.uniform(0.1, 0.5) * t)
        sig += drift

    sig = sig / (np.max(np.abs(sig)) + 1e-8)
    return sig.astype(np.float32)


def build_synthetic_corpus(n_per_class: int, fs: int, duration_s: float, seed: int = SEED) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    X, y = [], []
    for cls in range(CFG.n_classes):
        for _ in range(n_per_class):
            X.append(_synthesize_heart_sound(duration_s, fs, cls, rng))
            y.append(cls)
    return np.stack(X), np.array(y)


def load_physionet2016(data_dir: str, fs: int, duration_s: float) -> Tuple[np.ndarray, np.ndarray]:
    """Loads real PhysioNet 2016 data if `data_dir` exists, else raises FileNotFoundError."""
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(data_dir)

    signals, labels = [], []
    seg_len = int(duration_s * fs)
    subfolders = [d for d in sorted(os.listdir(data_dir)) if os.path.isdir(os.path.join(data_dir, d))]
    for sub in subfolders:
        sub_path = os.path.join(data_dir, sub)
        ref_path = os.path.join(sub_path, "REFERENCE.csv")
        if not os.path.exists(ref_path):
            continue
        ref = pd.read_csv(ref_path, header=None, names=["filename", "label"])
        for _, row in ref.iterrows():
            wav_path = os.path.join(sub_path, f"{row['filename']}.wav")
            if not os.path.exists(wav_path):
                continue
            orig_fs, data = wavfile.read(wav_path)
            data = data.astype(np.float32)
            data = data / (np.max(np.abs(data)) + 1e-8)
            if orig_fs != fs:
                n_samples_target = int(len(data) * fs / orig_fs)
                data = sps.resample(data, n_samples_target)
            # segment into fixed windows (drop remainder)
            for start in range(0, len(data) - seg_len + 1, seg_len):
                seg = data[start:start + seg_len]
                signals.append(seg)
                # 2-class fallback mapped into 5-class space: -1 normal -> 0, 1 abnormal -> generic 'Other' (4)
                labels.append(0 if row["label"] == -1 else 4)
    return np.stack(signals), np.array(labels)


print("Attempting to load real PhysioNet 2016 data from:", CFG.data_dir)
try:
    X_raw, y_raw = load_physionet2016(CFG.data_dir, CFG.sample_rate, CFG.segment_seconds)
    print(f"Loaded real dataset: {X_raw.shape[0]} segments.")
except FileNotFoundError:
    print("Real dataset not found -- generating a synthetic PCG corpus for demonstration.")
    X_raw, y_raw = build_synthetic_corpus(n_per_class=140, fs=CFG.sample_rate, duration_s=CFG.segment_seconds)
    print(f"Synthetic dataset built: {X_raw.shape[0]} segments, "
          f"{X_raw.shape[1]} samples/segment @ {CFG.sample_rate} Hz.")

print("Class distribution:", {CFG.class_names[c]: int((y_raw == c).sum()) for c in range(CFG.n_classes)})


In [ ]:
# Quick sanity-check plot: one normal vs. one abnormal (murmur) waveform -- mirrors paper Fig. 4
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
idx_normal = np.where(y_raw == 0)[0][0]
idx_abn = np.where(y_raw == 1)[0][0] if (y_raw == 1).any() else np.where(y_raw != 0)[0][0]
t_axis = np.arange(X_raw.shape[1]) / CFG.sample_rate
axes[0].plot(t_axis, X_raw[idx_normal], color="tab:blue")
axes[0].set_title("Normal PCG (raw)")
axes[1].plot(t_axis, X_raw[idx_abn], color="tab:red")
axes[1].set_title(f"Abnormal PCG (raw) -- class={CFG.class_names[y_raw[idx_abn]]}")
for ax in axes:
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
plt.tight_layout(); plt.show()


## 3. Variational Mode Decomposition (VMD) — Adaptive Signal Enhancement

Implements Eqs. (2)–(7): the signal $x(t)$ is decomposed into $K$ band-limited intrinsic mode functions (IMFs) $u_k(t)$ by solving the constrained variational problem via ADMM in the frequency domain. The enhanced signal $X_e(t)$ is reconstructed from the IMFs whose spectral energy is dominated by the cardiac frequency band (typically 20–200 Hz for PCG), discarding the noise-dominated modes.

In [ ]:
def vmd(signal: np.ndarray, alpha: float, tau: float, K: int, tol: float = 1e-7,
        max_iter: int = 300, dc: bool = False, init_omega: str = "uniform") -> Tuple[np.ndarray, np.ndarray]:
    """Variational Mode Decomposition (Dragomiretskiy & Zosso, 2014), implementing Eqs. (2)-(7).

    Returns
    -------
    u   : (K, N) array of the K decomposed IMFs in the time domain
    omega : (K,) array of the converged central frequencies (normalized, 0-0.5)
    """
    N = len(signal)
    # mirror the signal at both ends to reduce boundary effects
    f_mirror = np.concatenate([signal[N // 2 - 1::-1], signal, signal[:N // 2 - 1:-1]]) if N > 1 else signal
    T = len(f_mirror)
    t = np.arange(1, T + 1) / T
    freqs = t - 0.5 - 1.0 / T

    f_hat = np.fft.fftshift(np.fft.fft(f_mirror))
    f_hat_plus = f_hat.copy()
    f_hat_plus[:T // 2] = 0

    u_hat_plus = np.zeros((max_iter, len(freqs), K), dtype=complex)
    omega_plus = np.zeros((max_iter, K))

    if init_omega == "uniform":
        for k in range(K):
            omega_plus[0, k] = 0.5 * k / K
    else:
        omega_plus[0, :] = np.sort(np.exp(np.log(1 / T) + (np.log(0.5) - np.log(1 / T)) * np.random.rand(K)))

    if dc:
        omega_plus[0, 0] = 0

    lambda_hat = np.zeros((max_iter, len(freqs)), dtype=complex)
    uDiff = tol + np.finfo(float).eps
    n = 0

    while uDiff > tol and n < max_iter - 1:
        # update first mode
        sum_uk = u_hat_plus[n, :, K - 1] + np.sum(u_hat_plus[n, :, :K - 1], axis=1) - u_hat_plus[n, :, 0]
        u_hat_plus[n + 1, :, 0] = (f_hat_plus - sum_uk - lambda_hat[n] / 2) / (1 + alpha * (freqs - omega_plus[n, 0]) ** 2)

        if not dc:
            omega_plus[n + 1, 0] = (freqs[T // 2:] @ (np.abs(u_hat_plus[n + 1, T // 2:, 0]) ** 2)) / \
                                    (np.sum(np.abs(u_hat_plus[n + 1, T // 2:, 0]) ** 2) + np.finfo(float).eps)

        for k in range(1, K):
            sum_uk = sum_uk + u_hat_plus[n + 1, :, k - 1] - u_hat_plus[n, :, k]
            u_hat_plus[n + 1, :, k] = (f_hat_plus - sum_uk - lambda_hat[n] / 2) / (1 + alpha * (freqs - omega_plus[n, k]) ** 2)
            omega_plus[n + 1, k] = (freqs[T // 2:] @ (np.abs(u_hat_plus[n + 1, T // 2:, k]) ** 2)) / \
                                    (np.sum(np.abs(u_hat_plus[n + 1, T // 2:, k]) ** 2) + np.finfo(float).eps)

        lambda_hat[n + 1] = lambda_hat[n] + tau * (np.sum(u_hat_plus[n + 1], axis=1) - f_hat_plus)
        n += 1
        uDiff = np.finfo(float).eps
        for k in range(K):
            uDiff += (1.0 / T) * (u_hat_plus[n, :, k] - u_hat_plus[n - 1, :, k]) @ \
                     np.conj(u_hat_plus[n, :, k] - u_hat_plus[n - 1, :, k])
        uDiff = np.abs(uDiff)

    N_iter = n + 1
    omega = omega_plus[:N_iter, :]

    u_hat = np.zeros((T, K), dtype=complex)
    u_hat[T // 2:, :] = u_hat_plus[N_iter - 1, T // 2:, :]
    u_hat[1:T // 2 + 1, :] = np.conj(u_hat_plus[N_iter - 1, T // 2:, :][::-1, :])
    u_hat[0, :] = np.conj(u_hat[-1, :])

    u = np.zeros((K, T))
    for k in range(K):
        u[k] = np.real(np.fft.ifft(np.fft.ifftshift(u_hat[:, k])))

    # remove mirror padding
    u = u[:, T // 4: 3 * T // 4] if N > 1 else u
    u = u[:, :N]
    return u, omega[-1]


def vmd_enhance(signal: np.ndarray, fs: int, cfg: Config,
                 cardiac_band_hz: Tuple[float, float] = (15, 250)) -> np.ndarray:
    """Runs VMD and reconstructs the enhanced signal X_e(t) from the cardiac-relevant IMFs (Eq. 3 reconstruction),
    discarding modes whose central frequency falls outside the physiological PCG band.
    """
    imfs, omega = vmd(signal, alpha=cfg.vmd_alpha, tau=cfg.vmd_tau, K=cfg.vmd_K,
                       tol=cfg.vmd_tol, max_iter=cfg.vmd_max_iter)
    center_freqs_hz = omega * fs
    keep = (center_freqs_hz >= cardiac_band_hz[0]) & (center_freqs_hz <= cardiac_band_hz[1])
    if not keep.any():
        keep[np.argmin(center_freqs_hz)] = True  # always keep at least the lowest mode
    enhanced = imfs[keep].sum(axis=0)
    enhanced = enhanced / (np.max(np.abs(enhanced)) + 1e-8)
    return enhanced.astype(np.float32)


In [ ]:
# Demonstration: enhance one signal and visualize (paper Fig. 3 / Fig. 4 style)
demo_sig = X_raw[idx_abn]
demo_enh = vmd_enhance(demo_sig, CFG.sample_rate, CFG)

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
axes[0].plot(t_axis, demo_sig, color="tab:gray"); axes[0].set_title("Raw PCG")
axes[1].plot(t_axis, demo_enh, color="tab:green"); axes[1].set_title("VMD-Enhanced PCG")
axes[1].set_xlabel("Time (s)")
plt.tight_layout(); plt.show()


## 4. Multi-Resolution Wavelet Analysis (MRWA) — Hierarchical Feature Extraction

Implements Eqs. (8)–(13): wavelet-packet decomposition of the enhanced signal into $2^{J}$ sub-bands, followed by per-sub-band **energy** (Eq. 11) and **entropy** (Eq. 12) features, fused into $F_w$ (Eq. 13). A sequence of these fused vectors (one per short frame) forms the token sequence fed to MSAWT.

In [ ]:
def mrwa_extract(signal: np.ndarray, wavelet: str, level: int,
                   frame_len: int = 256, hop: int = 128) -> np.ndarray:
    """Frame-wise multi-resolution wavelet-packet feature extraction.

    For each analysis frame, decomposes with a wavelet packet tree of the given `level`
    (2**level sub-bands), then computes per-sub-band energy (Eq. 11) and Shannon entropy (Eq. 12),
    concatenated into the fused feature vector F_w (Eq. 13).

    Returns
    -------
    features : (n_frames, 2 * 2**level) array  -- [energy_1..energy_M, entropy_1..entropy_M]
    """
    if not HAVE_PYWT:
        raise ImportError("PyWavelets is required for MRWA -- `pip install PyWavelets`.")

    n = len(signal)
    frames = []
    for start in range(0, max(n - frame_len, 1), hop):
        frames.append(signal[start:start + frame_len])
    if not frames:
        frames = [signal]

    feats = []
    for fr in frames:
        if len(fr) < frame_len:
            fr = np.pad(fr, (0, frame_len - len(fr)))
        wp = pywt.WaveletPacket(data=fr, wavelet=wavelet, mode="symmetric", maxlevel=level)
        nodes = [node.path for node in wp.get_level(level, order="freq")]
        energies, probs = [], []
        for path in nodes:
            coeffs = wp[path].data
            e = float(np.sum(coeffs ** 2))          # Eq. (11)
            energies.append(e)
        total_e = sum(energies) + 1e-12
        for e in energies:
            probs.append(e / total_e)
        entropy = -sum(p * math.log(p + 1e-12) for p in probs)  # Eq. (12) (single scalar per frame)
        # spread entropy contribution evenly is not physical; instead report per-band local energy-entropy proxy
        entropies = [-(p * math.log(p + 1e-12)) for p in probs]
        feats.append(np.array(energies + entropies, dtype=np.float32))
    return np.stack(feats)  # (n_frames, 2 * n_subbands)


def mrwa_feature_dim(level: int) -> int:
    return 2 * (2 ** level)

print("MRWA feature dimension per frame:", mrwa_feature_dim(CFG.wp_level))
demo_feats = mrwa_extract(demo_enh, CFG.wavelet, CFG.wp_level)
print("Extracted MRWA feature-sequence shape (frames x features):", demo_feats.shape)


In [ ]:
# Visualize per-band energy as a spectrogram-like map (paper Fig. 6 style)
plt.figure(figsize=(8, 3.5))
n_sub = 2 ** CFG.wp_level
plt.imshow(demo_feats[:, :n_sub].T, aspect="auto", origin="lower", cmap="magma")
plt.colorbar(label="Sub-band energy")
plt.xlabel("Frame index"); plt.ylabel("Wavelet sub-band")
plt.title("MRWA Sub-band Energy Map (Spectrogram-like)")
plt.tight_layout(); plt.show()


## 5. Preprocessing Pipeline & PyTorch Dataset

Runs VMD $\to$ MRWA on every recording once (cached to disk-friendly numpy arrays), then wraps the resulting fixed-length feature-token sequences in a `torch.utils.data.Dataset`.

In [ ]:
def preprocess_corpus(X_raw: np.ndarray, cfg: Config, use_vmd: bool = True, use_mrwa: bool = True,
                        max_frames: int = 32) -> np.ndarray:
    """Applies VMD (optional, for ablation) and MRWA (optional) to every raw signal.
    Returns a fixed-length (N, max_frames, feat_dim) tensor, zero-padded / truncated per-recording.
    """
    feat_dim = mrwa_feature_dim(cfg.wp_level) if use_mrwa else 1
    out = np.zeros((len(X_raw), max_frames, feat_dim), dtype=np.float32)
    for i, sig in enumerate(X_raw):
        s = vmd_enhance(sig, cfg.sample_rate, cfg) if use_vmd else sig
        if use_mrwa:
            f = mrwa_extract(s, cfg.wavelet, cfg.wp_level)
        else:
            # raw-signal fallback token sequence (simple framing + RMS) for the "Without MRWA" ablation
            frame_len, hop = 256, 128
            frames = [s[j:j + frame_len] for j in range(0, max(len(s) - frame_len, 1), hop)]
            f = np.array([[np.sqrt(np.mean(fr ** 2))] for fr in frames], dtype=np.float32)
        n = min(len(f), max_frames)
        out[i, :n, :] = f[:n]
    return out


class PCGFeatureDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


print("Running VMD + MRWA over the full corpus (this may take a while for large datasets)...")
FEATURES = preprocess_corpus(X_raw, CFG, use_vmd=True, use_mrwa=True, max_frames=32)
LABELS = y_raw.copy()
print("Feature tensor shape:", FEATURES.shape)  # (N, max_frames, feat_dim)

FEAT_DIM = FEATURES.shape[-1]
MAX_FRAMES = FEATURES.shape[1]

# Normalize features (per-channel z-score) -- fit on train split only, applied globally below after split


In [ ]:
X_train_idx, X_temp_idx = train_test_split(
    np.arange(len(LABELS)), test_size=0.30, random_state=SEED, stratify=LABELS)
X_val_idx, X_test_idx = train_test_split(
    X_temp_idx, test_size=0.50, random_state=SEED, stratify=LABELS[X_temp_idx])

mu = FEATURES[X_train_idx].reshape(-1, FEAT_DIM).mean(axis=0)
sigma = FEATURES[X_train_idx].reshape(-1, FEAT_DIM).std(axis=0) + 1e-6
FEATURES_NORM = (FEATURES - mu) / sigma

train_ds = PCGFeatureDataset(FEATURES_NORM[X_train_idx], LABELS[X_train_idx])
val_ds = PCGFeatureDataset(FEATURES_NORM[X_val_idx], LABELS[X_val_idx])
test_ds = PCGFeatureDataset(FEATURES_NORM[X_test_idx], LABELS[X_test_idx])

print(f"Train/Val/Test sizes: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")


## 6. Conditional Diffusion Probabilistic Model (CDPM) — Synthetic Abnormal-Sample Generation

Implements Eqs. (14)–(23). The diffusion model operates directly on the (flattened) MRWA feature sequence of a recording, conditioned on the target class label $c$. It is trained with the standard denoising-noise-prediction objective $\mathcal{L}_{diff}$ (Eq. 21) plus a feature-consistency term $\mathcal{L}_{con}$ (Eq. 22), and used to synthesize additional minority-class (abnormal) samples that are appended to the training set (Eq. 23, $D_{aug} = D_{real} \cup D_{synthetic}$).

In [ ]:
def make_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    return torch.linspace(beta_start, beta_end, T)


class ConditionalDenoiser(nn.Module):
    """epsilon_theta(x_t, t, c) -- predicts the noise added at diffusion step t, conditioned on class c."""
    def __init__(self, feat_dim: int, n_classes: int, hidden: int = 256, T: int = 200):
        super().__init__()
        self.feat_dim = feat_dim
        self.time_embed = nn.Sequential(
            nn.Embedding(T, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.class_embed = nn.Embedding(n_classes, hidden)
        self.net = nn.Sequential(
            nn.Linear(feat_dim + hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, feat_dim)
        )

    def forward(self, x_t, t, c):
        te = self.time_embed(t)
        ce = self.class_embed(c)
        h = te + ce
        inp = torch.cat([x_t, h], dim=-1)
        return self.net(inp)


class CDPM:
    """Conditional Diffusion Probabilistic Model over flattened MRWA feature vectors."""
    def __init__(self, feat_dim: int, n_classes: int, cfg: Config, device):
        self.T = cfg.diffusion_timesteps
        self.device = device
        self.betas = make_beta_schedule(self.T).to(device)          # beta_t, Eq. (14)
        self.alphas = 1.0 - self.betas                               # alpha_t, Eq. (16)
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)          # \bar{alpha}_t, Eq. (17)
        self.model = ConditionalDenoiser(feat_dim, n_classes, cfg.diffusion_hidden, self.T).to(device)
        self.lambda_con = cfg.diffusion_lambda_con

    def q_sample(self, x0, t, noise=None):
        """Forward diffusion, Eq. (15): x_t = sqrt(alpha_bar_t) x0 + sqrt(1-alpha_bar_t) * noise."""
        if noise is None:
            noise = torch.randn_like(x0)
        ab = self.alpha_bars[t].unsqueeze(-1)
        return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise, noise

    def loss(self, x0, c):
        b = x0.shape[0]
        t = torch.randint(0, self.T, (b,), device=self.device)
        x_t, noise = self.q_sample(x0, t)
        noise_pred = self.model(x_t, t, c)
        l_diff = F.mse_loss(noise_pred, noise)                       # Eq. (21)
        # feature-consistency loss: predicted x0 reconstruction should stay close to true x0 (Eq. 22 support)
        ab = self.alpha_bars[t].unsqueeze(-1)
        x0_pred = (x_t - torch.sqrt(1 - ab) * noise_pred) / torch.sqrt(ab)
        l_con = F.mse_loss(x0_pred, x0)
        return l_diff + self.lambda_con * l_con                      # Eq. (22)

    @torch.no_grad()
    def sample(self, n_samples: int, feat_dim: int, class_label: int):
        """Reverse diffusion sampling, Eq. (18)-(20)."""
        x = torch.randn(n_samples, feat_dim, device=self.device)
        c = torch.full((n_samples,), class_label, dtype=torch.long, device=self.device)
        for step in reversed(range(self.T)):
            t = torch.full((n_samples,), step, dtype=torch.long, device=self.device)
            eps_pred = self.model(x, t, c)
            alpha_t = self.alphas[step]
            alpha_bar_t = self.alpha_bars[step]
            beta_t = self.betas[step]
            z = torch.randn_like(x) if step > 0 else torch.zeros_like(x)
            sigma_t = torch.sqrt(beta_t)
            x = (1.0 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * eps_pred) + sigma_t * z
        return x


def train_cdpm(features: np.ndarray, labels: np.ndarray, cfg: Config, device) -> CDPM:
    flat_dim = features.shape[1] * features.shape[2]
    x_flat = torch.tensor(features.reshape(len(features), flat_dim), dtype=torch.float32, device=device)
    y = torch.tensor(labels, dtype=torch.long, device=device)

    cdpm = CDPM(flat_dim, cfg.n_classes, cfg, device)
    opt = torch.optim.Adam(cdpm.model.parameters(), lr=cfg.diffusion_lr)

    n = len(y)
    for epoch in range(cfg.diffusion_epochs):
        perm = torch.randperm(n)
        total_loss = 0.0
        for i in range(0, n, cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            xb, cb = x_flat[idx], y[idx]
            loss = cdpm.loss(xb, cb)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * len(idx)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[CDPM] epoch {epoch+1}/{cfg.diffusion_epochs}  loss={total_loss/n:.5f}")
    return cdpm


def augment_with_synthetic(features, labels, cdpm: CDPM, cfg: Config, device,
                             target_per_class: Optional[int] = None):
    """Balances the dataset by synthesizing extra samples for under-represented classes (Eq. 23)."""
    feat_dim = features.shape[1] * features.shape[2]
    counts = {c: int((labels == c).sum()) for c in range(cfg.n_classes)}
    target = target_per_class or max(counts.values())

    synth_X, synth_y = [], []
    for c, cnt in counts.items():
        n_needed = target - cnt
        if n_needed <= 0:
            continue
        gen = cdpm.sample(n_needed, feat_dim, c).cpu().numpy()
        gen = gen.reshape(n_needed, features.shape[1], features.shape[2])
        synth_X.append(gen)
        synth_y.append(np.full(n_needed, c))

    if synth_X:
        synth_X = np.concatenate(synth_X, axis=0)
        synth_y = np.concatenate(synth_y, axis=0)
        aug_X = np.concatenate([features, synth_X], axis=0)     # D_aug = D_real U D_synthetic
        aug_y = np.concatenate([labels, synth_y], axis=0)
    else:
        aug_X, aug_y = features, labels
    return aug_X, aug_y


In [ ]:
print("Training CDPM on the training split...")
cdpm = train_cdpm(FEATURES_NORM[X_train_idx], LABELS[X_train_idx], CFG, DEVICE)

train_X_aug, train_y_aug = augment_with_synthetic(
    FEATURES_NORM[X_train_idx], LABELS[X_train_idx], cdpm, CFG, DEVICE)
print("Balanced training set size after CDPM augmentation:", len(train_y_aug))
print("New class distribution:", {CFG.class_names[c]: int((train_y_aug == c).sum()) for c in range(CFG.n_classes)})

train_ds_aug = PCGFeatureDataset(train_X_aug, train_y_aug)


## 7. Multi-Head Self-Attention Wavelet Transformer (MSAWT)

Implements Eqs. (24)–(37): wavelet-feature embedding + positional encoding, stacked multi-head self-attention encoder blocks with residual connections & layer norm, and a feed-forward refinement stage.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerEncoderBlock(nn.Module):
    """Implements Eqs. (27)-(35): MHSA -> residual+LN -> FFN -> residual+LN."""
    def __init__(self, d_model: int, n_heads: int, ff_hidden: int, dropout: float):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_hidden), nn.ReLU(), nn.Linear(ff_hidden, d_model))
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, z, attn_mask=None):
        attn_out, _ = self.mha(z, z, z, attn_mask=attn_mask, need_weights=False)   # Eqs. (27)-(30)
        z = self.ln1(z + self.dropout(attn_out))                                    # Eq. (34)
        ffn_out = self.ffn(z)                                                       # Eq. (33)
        z = self.ln2(z + self.dropout(ffn_out))                                     # Eq. (35)
        return z


class MSAWT(nn.Module):
    """Multi-Head Self-Attention Wavelet Transformer -- Eqs. (24)-(36)."""
    def __init__(self, feat_dim: int, cfg: Config):
        super().__init__()
        self.embed = nn.Linear(feat_dim, cfg.d_model)        # W_e, b_e  (Eq. 25)
        self.pos_enc = PositionalEncoding(cfg.d_model)         # P         (Eq. 26)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(cfg.d_model, cfg.n_heads, cfg.ff_hidden, cfg.dropout)
            for _ in range(cfg.n_layers)
        ])
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        # x: (batch, n_frames, feat_dim)
        z = self.embed(x)                 # Eq. (25)
        z = self.pos_enc(z)                # Eq. (26)
        z = self.dropout(z)
        for block in self.blocks:
            z = block(z)                   # Eqs. (27)-(35), stacked
        # global cardiac representation: mean-pool over the frame/token dimension
        f_T = z.mean(dim=1)                # F_T,  Eq. (36)
        return f_T


## 8. ACACC — Adaptive Confidence-Aware Cardiac Abnormality Classification

Implements Eqs. (48)–(57): a linear classification head over $F_T$, Softmax posterior, confidence score $\text{Conf} = \max_i P(y_i)$, cross-entropy loss with L2 regularization on the classifier weights.

In [ ]:
class ACACC(nn.Module):
    def __init__(self, d_model: int, n_classes: int):
        super().__init__()
        self.classifier = nn.Linear(d_model, n_classes)   # W_c, b_c  (Eq. 49)

    def forward(self, f_t):
        logits = self.classifier(f_t)                     # Z          (Eq. 49)
        probs = F.softmax(logits, dim=-1)                  # P(y_i)     (Eq. 50)
        pred = torch.argmax(probs, dim=-1)                 # y_hat      (Eq. 51)
        conf = torch.max(probs, dim=-1).values              # Conf       (Eq. 52)
        return logits, probs, pred, conf


class FSMWDDS(nn.Module):
    """Full pipeline head: MSAWT feature learner + ACACC classifier."""
    def __init__(self, feat_dim: int, cfg: Config):
        super().__init__()
        self.msawt = MSAWT(feat_dim, cfg)
        self.acacc = ACACC(cfg.d_model, cfg.n_classes)

    def forward(self, x):
        f_t = self.msawt(x)
        logits, probs, pred, conf = self.acacc(x=None) if False else self.acacc(f_t)
        return logits, probs, pred, conf

    def acacc_loss(self, logits, y, l2_lambda: float):
        """Eq. (53)-(55): categorical cross-entropy + L2 weight regularization."""
        l_cls = F.cross_entropy(logits, y)
        l_reg = l2_lambda * torch.sum(self.acacc.classifier.weight ** 2)
        return l_cls + l_reg


## 9. FedProx — Privacy-Preserving Federated Optimization

Implements Eqs. (38)–(47). The (augmented) training set is partitioned across `cfg.n_clients` simulated healthcare institutions (non-IID split by class-skewed sampling, to reflect realistic hospital heterogeneity). Each communication round:

1. The server broadcasts global weights $w_g^{(t)}$ to every client.
2. Each client runs local SGD for `local_epochs`, minimizing $\mathcal{L}_k(w) = F_k(w) + \frac{\mu}{2}\lVert w - w_g \rVert^2$ (Eq. 40, the proximal term keeps local updates from drifting too far from the global model under non-IID data).
3. The server aggregates via weighted averaging $w_g^{(t+1)} = \sum_k \frac{n_k}{n} w_k^{(t+1)}$ (Eq. 42).
4. Only model **parameters** are ever transmitted — raw PCG data never leaves its institution (Eqs. 46–47).

In [ ]:
def partition_non_iid(n_samples: int, labels: np.ndarray, n_clients: int, skew: float = 0.6, seed: int = SEED):
    """Dirichlet-like class-skewed partition so clients have heterogeneous (non-IID) label distributions,
    reflecting real multi-hospital data heterogeneity.
    """
    rng = np.random.default_rng(seed)
    idx_by_class = {c: np.where(labels == c)[0] for c in np.unique(labels)}
    for c in idx_by_class:
        rng.shuffle(idx_by_class[c])

    client_indices = [[] for _ in range(n_clients)]
    for c, idxs in idx_by_class.items():
        # skewed split: each client gets a Dirichlet-weighted share of this class
        weights = rng.dirichlet(alpha=[1 - skew + 0.1] * n_clients)
        splits = (np.cumsum(weights) * len(idxs)).astype(int)[:-1]
        parts = np.split(idxs, splits)
        for k in range(n_clients):
            client_indices[k].extend(parts[k].tolist())

    return [np.array(sorted(ci)) for ci in client_indices]


def get_flat_params(model: nn.Module) -> torch.Tensor:
    return torch.cat([p.data.view(-1) for p in model.parameters()])


def set_flat_params(model: nn.Module, flat: torch.Tensor):
    offset = 0
    for p in model.parameters():
        numel = p.numel()
        p.data.copy_(flat[offset:offset + numel].view_as(p))
        offset += numel


def local_train_fedprox(local_model: nn.Module, global_params: torch.Tensor, loader: DataLoader,
                          cfg: Config, device) -> nn.Module:
    local_model.train()
    opt = torch.optim.Adam(local_model.parameters(), lr=cfg.client_lr, weight_decay=cfg.weight_decay)
    for _ in range(cfg.local_epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, probs, pred, conf = local_model(xb)
            loss = local_model.acacc_loss(logits, yb, cfg.l2_reg)      # F_k(w),   Eq. (40) first term
            prox = 0.0
            flat_local = get_flat_params(local_model)
            prox = (cfg.fedprox_mu / 2.0) * torch.sum((flat_local - global_params) ** 2)  # proximal term
            total_loss = loss + prox
            opt.zero_grad(); total_loss.backward(); opt.step()
    return local_model


def fedprox_train(train_X, train_y, val_ds, feat_dim: int, cfg: Config, device,
                    log_metrics: bool = True):
    client_idx = partition_non_iid(len(train_y), train_y, cfg.n_clients)
    client_sizes = [len(ci) for ci in client_idx]
    n_total = sum(client_sizes)

    global_model = FSMWDDS(feat_dim, cfg).to(device)
    history = {"accuracy": [], "precision": [], "recall": [], "specificity": [], "f1": [], "mcc": []}

    for rnd in range(cfg.comm_rounds):
        global_flat = get_flat_params(global_model).clone()
        local_flats = []

        for k in range(cfg.n_clients):
            if client_sizes[k] == 0:
                continue
            local_model = FSMWDDS(feat_dim, cfg).to(device)
            set_flat_params(local_model, global_flat)     # broadcast w_g^t -> w_k^t   (Eq. 44)

            ds_k = PCGFeatureDataset(train_X[client_idx[k]], train_y[client_idx[k]])
            loader_k = DataLoader(ds_k, batch_size=cfg.batch_size, shuffle=True)

            local_model = local_train_fedprox(local_model, global_flat, loader_k, cfg, device)
            local_flats.append((get_flat_params(local_model), client_sizes[k]))

        # weighted aggregation, Eq. (42)
        agg = torch.zeros_like(global_flat)
        for flat_k, n_k in local_flats:
            agg += (n_k / n_total) * flat_k
        set_flat_params(global_model, agg)

        if log_metrics and ((rnd + 1) % 1 == 0):
            m = evaluate_model(global_model, val_ds, device, cfg)
            for key in history:
                history[key].append(m[key])
            if (rnd + 1) % 5 == 0 or rnd == 0:
                print(f"[FedProx] round {rnd+1:3d}/{cfg.comm_rounds}  "
                      f"acc={m['accuracy']*100:.2f}%  prec={m['precision']*100:.2f}%  "
                      f"rec={m['recall']*100:.2f}%  f1={m['f1']*100:.2f}%  mcc={m['mcc']:.3f}")

    return global_model, history


## 10. Evaluation Metrics

Implements the full metric suite reported in the paper: Accuracy, Precision, Recall (Sensitivity), Specificity, F1-score, MCC, NPV, Balanced Accuracy, Cohen's Kappa, G-Mean, Jaccard Index, Youden's Index, FPR, FNR, Diagnostic Odds Ratio (DOR), and AUPRC (macro, one-vs-rest).

In [ ]:
def compute_full_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_probs: np.ndarray, n_classes: int) -> Dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    tp = np.diag(cm).astype(float)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    tn = cm.sum() - (tp + fp + fn)

    with np.errstate(divide="ignore", invalid="ignore"):
        specificity_per_class = np.nan_to_num(tn / (tn + fp))
        npv_per_class = np.nan_to_num(tn / (tn + fn))
        fpr_per_class = np.nan_to_num(fp / (fp + tn))
        fnr_per_class = np.nan_to_num(fn / (fn + tp))
        sensitivity_per_class = np.nan_to_num(tp / (tp + fn))
        dor_per_class = np.nan_to_num((tp * tn) / (fp * fn + 1e-9))

    specificity = float(np.mean(specificity_per_class))
    npv = float(np.mean(npv_per_class))
    fpr = float(np.mean(fpr_per_class))
    fnr = float(np.mean(fnr_per_class))
    dor = float(np.mean(dor_per_class[np.isfinite(dor_per_class)])) if np.isfinite(dor_per_class).any() else float("nan")
    youden = float(np.mean(sensitivity_per_class + specificity_per_class - 1))
    g_mean = float(np.sqrt(np.mean(sensitivity_per_class) * specificity))

    try:
        auprc = average_precision_score(
            np.eye(n_classes)[y_true], y_probs, average="macro")
    except Exception:
        auprc = float("nan")

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "npv": npv,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
        "g_mean": g_mean,
        "jaccard": jaccard_score(y_true, y_pred, average="macro", zero_division=0),
        "youden_index": youden,
        "fpr": fpr,
        "fnr": fnr,
        "dor": dor,
        "auprc": auprc,
    }
    return metrics


@torch.no_grad()
def evaluate_model(model: nn.Module, dataset: Dataset, device, cfg: Config) -> Dict[str, float]:
    model.eval()
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False)
    all_y, all_pred, all_probs = [], [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits, probs, pred, conf = model(xb)
        all_y.append(yb.numpy())
        all_pred.append(pred.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    y_probs = np.concatenate(all_probs)
    return compute_full_metrics(y_true, y_pred, y_probs, cfg.n_classes)


def pretty_print_metrics(m: Dict[str, float]):
    order = ["accuracy", "precision", "recall", "specificity", "f1", "mcc", "npv",
             "balanced_accuracy", "cohen_kappa", "g_mean", "jaccard", "youden_index",
             "fpr", "fnr", "dor", "auprc"]
    for k in order:
        v = m[k]
        if k == "mcc":
            print(f"  {k:20s}: {v:.4f}")
        elif k == "dor":
            print(f"  {k:20s}: {v:.2f}")
        else:
            print(f"  {k:20s}: {v*100:.2f}%")


## 11. Train the Full FSMW-DDS Model (Federated, with CDPM Augmentation)

In [ ]:
print("="*70)
print("Training the complete FSMW-DDS framework (VMD + MRWA + CDPM + MSAWT + FedProx + ACACC)")
print("="*70)

full_model, full_history = fedprox_train(train_X_aug, train_y_aug, val_ds, FEAT_DIM, CFG, DEVICE)

print("\nFinal Test-set performance:")
test_metrics_full = evaluate_model(full_model, test_ds, DEVICE, CFG)
pretty_print_metrics(test_metrics_full)


In [ ]:
# Training curves (paper Figs. 12-17 style)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
metric_keys = ["accuracy", "precision", "recall", "specificity", "f1", "mcc"]
titles = ["Accuracy", "Precision", "Recall", "Specificity", "F1-score", "MCC"]
rounds_axis = np.arange(1, len(full_history["accuracy"]) + 1)

for ax, key, title in zip(axes.flat, metric_keys, titles):
    y = np.array(full_history[key])
    y_plot = y * 100 if key != "mcc" else y
    ax.plot(rounds_axis, y_plot, color="tab:blue")
    ax.set_title(f"{title} vs. Communication Round")
    ax.set_xlabel("Round"); ax.set_ylabel(title + (" (%)" if key != "mcc" else ""))
    ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## 12. Comparison with State-of-the-Art & Baseline Models

We provide (a) a lightweight re-implementation of the comparison baselines (1D CNN, CNN+LSTM, ResNet-1D, ViT-style encoder) trained on the *same* preprocessed features for a fair, reproducible comparison, and (b) a table of the paper's reported reference numbers for context. Swap in your own trained numbers once you run this on the real PhysioNet corpus.

In [ ]:
class Simple1DCNN(nn.Module):
    def __init__(self, feat_dim, n_frames, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(feat_dim, 64, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, T, F) -> (B, F, T)
        h = self.conv(x.transpose(1, 2)).squeeze(-1)
        logits = self.fc(h)
        probs = F.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1)
        conf = torch.max(probs, dim=-1).values
        return logits, probs, pred, conf

    def acacc_loss(self, logits, y, l2_lambda):
        return F.cross_entropy(logits, y)


class CNNLSTM(nn.Module):
    def __init__(self, feat_dim, n_frames, n_classes, hidden=128):
        super().__init__()
        self.conv = nn.Conv1d(feat_dim, 64, 3, padding=1)
        self.lstm = nn.LSTM(64, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, n_classes)

    def forward(self, x):
        h = F.relu(self.conv(x.transpose(1, 2))).transpose(1, 2)
        out, _ = self.lstm(h)
        pooled = out.mean(dim=1)
        logits = self.fc(pooled)
        probs = F.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1)
        conf = torch.max(probs, dim=-1).values
        return logits, probs, pred, conf

    def acacc_loss(self, logits, y, l2_lambda):
        return F.cross_entropy(logits, y)


class ResNet1DBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv1d(ch, ch, 3, padding=1)
        self.conv2 = nn.Conv1d(ch, ch, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(ch); self.bn2 = nn.BatchNorm1d(ch)

    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        return F.relu(x + h)


class ResNet1D(nn.Module):
    def __init__(self, feat_dim, n_frames, n_classes, ch=64, n_blocks=4):
        super().__init__()
        self.stem = nn.Conv1d(feat_dim, ch, 3, padding=1)
        self.blocks = nn.Sequential(*[ResNet1DBlock(ch) for _ in range(n_blocks)])
        self.fc = nn.Linear(ch, n_classes)

    def forward(self, x):
        h = F.relu(self.stem(x.transpose(1, 2)))
        h = self.blocks(h)
        pooled = h.mean(dim=-1)
        logits = self.fc(pooled)
        probs = F.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1)
        conf = torch.max(probs, dim=-1).values
        return logits, probs, pred, conf

    def acacc_loss(self, logits, y, l2_lambda):
        return F.cross_entropy(logits, y)


class ViTStyle(nn.Module):
    """Plain (non-wavelet-specific) transformer encoder baseline -- MSAWT without the wavelet framing/CDPM/FedProx pipeline around it."""
    def __init__(self, feat_dim, n_frames, n_classes, d_model=128, n_heads=8, n_layers=4):
        super().__init__()
        self.embed = nn.Linear(feat_dim, d_model)
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=256, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model, n_classes)

    def forward(self, x):
        h = self.pos(self.embed(x))
        h = self.encoder(h).mean(dim=1)
        logits = self.fc(h)
        probs = F.softmax(logits, dim=-1)
        pred = torch.argmax(probs, dim=-1)
        conf = torch.max(probs, dim=-1).values
        return logits, probs, pred, conf

    def acacc_loss(self, logits, y, l2_lambda):
        return F.cross_entropy(logits, y)


def train_plain(model, train_ds, val_ds, cfg, device, epochs=None):
    epochs = epochs or cfg.epochs
    loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    model.to(device)
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits, probs, pred, conf = model(xb)
            loss = model.acacc_loss(logits, yb, cfg.l2_reg)
            opt.zero_grad(); loss.backward(); opt.step()
    return model


In [ ]:
baseline_ctor = {
    "1D CNN": lambda: Simple1DCNN(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "CNN+LSTM": lambda: CNNLSTM(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "ResNet-1D": lambda: ResNet1D(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "ViT-style": lambda: ViTStyle(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
}

baseline_results = {}
for name, ctor in baseline_ctor.items():
    print(f"Training baseline: {name} ...")
    m = train_plain(ctor(), train_ds_aug, val_ds, CFG, DEVICE, epochs=25)
    baseline_results[name] = evaluate_model(m, test_ds, DEVICE, CFG)
    print(f"  -> accuracy={baseline_results[name]['accuracy']*100:.2f}%  "
          f"f1={baseline_results[name]['f1']*100:.2f}%  mcc={baseline_results[name]['mcc']:.3f}")

baseline_results["FSMW-DDS (Proposed)"] = test_metrics_full


In [ ]:
comparison_df = pd.DataFrame(baseline_results).T
display_cols = ["accuracy", "precision", "recall", "f1", "mcc", "npv",
                 "balanced_accuracy", "cohen_kappa", "g_mean", "jaccard",
                 "youden_index", "fpr", "fnr", "dor", "auprc"]
comparison_df = comparison_df[display_cols]
pct_cols = [c for c in display_cols if c not in ("mcc", "dor")]
comparison_df_display = comparison_df.copy()
comparison_df_display[pct_cols] = (comparison_df_display[pct_cols] * 100).round(2)
comparison_df_display["mcc"] = comparison_df_display["mcc"].round(4)
comparison_df_display["dor"] = comparison_df_display["dor"].round(2)
comparison_df_display


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
methods = list(baseline_results.keys())
f1_vals = [baseline_results[m]["f1"] * 100 for m in methods]
colors = ["tab:gray"] * (len(methods) - 1) + ["tab:red"]
bars = ax.bar(methods, f1_vals, color=colors)
ax.set_ylabel("F1-score (%)")
ax.set_title("F1-score: Proposed FSMW-DDS vs. Baseline / SOTA Models")
ax.set_ylim(0, 100)
plt.xticks(rotation=20, ha="right")
for b, v in zip(bars, f1_vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}", ha="center")
plt.tight_layout(); plt.show()


## 13. Ablation Study

Reproduces Table 4 of the paper: remove each module in turn (VMD, MRWA, CDPM, FedProx, ACACC-vs-plain-softmax) and measure the performance drop, using the same non-federated training loop for the non-FedProx-relevant ablations and the federated loop for the "Without FedProx" case (replaced by centralized training).

In [ ]:
def run_ablation(name: str, use_vmd: bool, use_mrwa: bool, use_cdpm: bool,
                   use_fedprox: bool, use_acacc: bool, cfg: Config, device,
                   epochs_centralized: int = 25) -> Dict[str, float]:
    print(f"\n--- Ablation: {name} ---")

    if use_vmd == True and use_mrwa == True:
        feats = FEATURES_NORM  # already computed with VMD+MRWA
    else:
        feats = preprocess_corpus(X_raw, cfg, use_vmd=use_vmd, use_mrwa=use_mrwa, max_frames=MAX_FRAMES)
        m2 = feats[X_train_idx].reshape(-1, feats.shape[-1]).mean(axis=0)
        s2 = feats[X_train_idx].reshape(-1, feats.shape[-1]).std(axis=0) + 1e-6
        feats = (feats - m2) / s2

    feat_dim_local = feats.shape[-1]
    tr_X, tr_y = feats[X_train_idx], LABELS[X_train_idx]

    if use_cdpm:
        cdpm_local = train_cdpm(tr_X, tr_y, cfg, device)
        tr_X, tr_y = augment_with_synthetic(tr_X, tr_y, cdpm_local, cfg, device)

    val_ds_local = PCGFeatureDataset(feats[X_val_idx], LABELS[X_val_idx])
    test_ds_local = PCGFeatureDataset(feats[X_test_idx], LABELS[X_test_idx])

    class ACACCLessModel(nn.Module):
        """Plain softmax head without confidence scoring / L2 emphasis, for the 'Without ACACC' ablation."""
        def __init__(self, feat_dim, cfg):
            super().__init__()
            self.msawt = MSAWT(feat_dim, cfg)
            self.fc = nn.Linear(cfg.d_model, cfg.n_classes)

        def forward(self, x):
            f_t = self.msawt(x)
            logits = self.fc(f_t)
            probs = F.softmax(logits, dim=-1)
            pred = torch.argmax(probs, dim=-1)
            conf = torch.max(probs, dim=-1).values
            return logits, probs, pred, conf

        def acacc_loss(self, logits, y, l2_lambda):
            return F.cross_entropy(logits, y)  # no L2 term -> ablates ACACC's regularized confidence-aware loss

    model_ctor = (lambda: FSMWDDS(feat_dim_local, cfg)) if use_acacc else (lambda: ACACCLessModel(feat_dim_local, cfg))

    if use_fedprox:
        trained_model, _ = fedprox_train(tr_X, tr_y, val_ds_local, feat_dim_local, cfg, device, log_metrics=False)
    else:
        train_ds_local = PCGFeatureDataset(tr_X, tr_y)
        trained_model = train_plain(model_ctor(), train_ds_local, val_ds_local, cfg, device, epochs=epochs_centralized)

    metrics = evaluate_model(trained_model, test_ds_local, device, cfg)
    print(f"  accuracy={metrics['accuracy']*100:.2f}%  precision={metrics['precision']*100:.2f}%  "
          f"recall={metrics['recall']*100:.2f}%  f1={metrics['f1']*100:.2f}%  mcc={metrics['mcc']:.3f}")
    return metrics


In [ ]:
# NOTE: for speed, the ablation study below uses a reduced number of FedProx rounds / centralized epochs.
# Increase `CFG.comm_rounds` / `epochs_centralized` for paper-scale reproduction.
ablation_cfg = Config(**{**CFG.__dict__, "comm_rounds": 15, "diffusion_epochs": 20})

ablation_results = {
    "Without VMD":     run_ablation("Without VMD", use_vmd=False, use_mrwa=True, use_cdpm=True, use_fedprox=True, use_acacc=True, cfg=ablation_cfg, device=DEVICE),
    "Without MRWA":    run_ablation("Without MRWA", use_vmd=True, use_mrwa=False, use_cdpm=True, use_fedprox=True, use_acacc=True, cfg=ablation_cfg, device=DEVICE),
    "Without CDPM":    run_ablation("Without CDPM", use_vmd=True, use_mrwa=True, use_cdpm=False, use_fedprox=True, use_acacc=True, cfg=ablation_cfg, device=DEVICE),
    "Without FedProx": run_ablation("Without FedProx", use_vmd=True, use_mrwa=True, use_cdpm=True, use_fedprox=False, use_acacc=True, cfg=ablation_cfg, device=DEVICE, epochs_centralized=25),
    "Without ACACC":   run_ablation("Without ACACC", use_vmd=True, use_mrwa=True, use_cdpm=True, use_fedprox=True, use_acacc=False, cfg=ablation_cfg, device=DEVICE),
}
ablation_results["Proposed FSMW-DDS (Complete)"] = test_metrics_full

ablation_df = pd.DataFrame(ablation_results).T[["accuracy", "precision", "recall", "f1", "mcc"]]
ablation_df[["accuracy", "precision", "recall", "f1"]] = (ablation_df[["accuracy", "precision", "recall", "f1"]] * 100).round(2)
ablation_df["mcc"] = ablation_df["mcc"].round(3)
ablation_df


## 14. 5-Fold Stratified Cross-Validation

In [ ]:
def run_kfold(features, labels, cfg, device, k=5, comm_rounds=15):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=SEED)
    fold_metrics = []
    kfold_cfg = Config(**{**cfg.__dict__, "comm_rounds": comm_rounds})

    for fold, (tr_idx, te_idx) in enumerate(skf.split(features, labels)):
        print(f"\n--- Fold {fold+1}/{k} ---")
        tr_idx_in, val_idx_in = train_test_split(tr_idx, test_size=0.15, random_state=SEED, stratify=labels[tr_idx])

        cdpm_fold = train_cdpm(features[tr_idx_in], labels[tr_idx_in], kfold_cfg, device)
        aug_X, aug_y = augment_with_synthetic(features[tr_idx_in], labels[tr_idx_in], cdpm_fold, kfold_cfg, device)

        val_ds_fold = PCGFeatureDataset(features[val_idx_in], labels[val_idx_in])
        test_ds_fold = PCGFeatureDataset(features[te_idx], labels[te_idx])

        model_fold, _ = fedprox_train(aug_X, aug_y, val_ds_fold, features.shape[-1], kfold_cfg, device, log_metrics=False)
        m = evaluate_model(model_fold, test_ds_fold, device, kfold_cfg)
        fold_metrics.append(m)
        print(f"  Fold {fold+1}: acc={m['accuracy']*100:.2f}%  f1={m['f1']*100:.2f}%  mcc={m['mcc']:.3f}")

    return fold_metrics


fold_results = run_kfold(FEATURES_NORM, LABELS, CFG, DEVICE, k=5, comm_rounds=10)

cv_table = pd.DataFrame(fold_results)[["accuracy", "precision", "recall", "f1", "mcc"]]
cv_table.index = [f"Fold {i+1}" for i in range(len(cv_table))]
cv_summary = cv_table.copy()
cv_summary.loc["Mean"] = cv_table.mean()
cv_summary.loc["Std. Dev."] = cv_table.std()
cv_summary[["accuracy", "precision", "recall", "f1"]] = (cv_summary[["accuracy", "precision", "recall", "f1"]] * 100).round(2)
cv_summary["mcc"] = cv_summary["mcc"].round(4)
cv_summary


## 15. Statistical Significance & Stability Analysis

Runs the complete FSMW-DDS pipeline across several independent seeds to (a) report mean/std/95% CI stability, and (b) perform a paired t-test of the proposed model's per-fold (or per-run) accuracy against each baseline.

In [ ]:
def run_independent_seeds(n_runs, cfg, device, comm_rounds=10):
    results = []
    for run in range(n_runs):
        seed_r = SEED + run
        torch.manual_seed(seed_r); np.random.seed(seed_r)
        run_cfg = Config(**{**cfg.__dict__, "comm_rounds": comm_rounds})
        cdpm_r = train_cdpm(FEATURES_NORM[X_train_idx], LABELS[X_train_idx], run_cfg, device)
        aug_X, aug_y = augment_with_synthetic(FEATURES_NORM[X_train_idx], LABELS[X_train_idx], cdpm_r, run_cfg, device)
        model_r, _ = fedprox_train(aug_X, aug_y, val_ds, FEAT_DIM, run_cfg, device, log_metrics=False)
        m = evaluate_model(model_r, test_ds, device, run_cfg)
        results.append(m)
        print(f"Run {run+1}/{n_runs}: acc={m['accuracy']*100:.2f}%  f1={m['f1']*100:.2f}%  mcc={m['mcc']:.3f}")
    return results


stability_runs = run_independent_seeds(5, CFG, DEVICE, comm_rounds=10)

stability_df = pd.DataFrame(stability_runs)[["accuracy", "precision", "recall", "f1", "mcc"]]
stability_df.index = [f"Run {i+1}" for i in range(len(stability_df))]

mean_row = stability_df.mean()
std_row = stability_df.std()
n = len(stability_df)
ci95 = 1.96 * std_row / np.sqrt(n)

stability_summary = stability_df.copy()
stability_summary.loc["Mean"] = mean_row
stability_summary.loc["Std. Dev."] = std_row
stability_summary.loc["95% CI (+/-)"] = ci95
stability_summary


In [ ]:
# Paired t-test: proposed model's per-run accuracy vs. each baseline's (single) test accuracy,
# using the proposed model's run-to-run distribution against the baseline's point estimate
# replicated across the same number of runs (one-sample t-test against the baseline accuracy).
ttest_rows = []
proposed_accs = stability_df["accuracy"].values
for name in ["1D CNN", "CNN+LSTM", "ResNet-1D", "ViT-style"]:
    baseline_acc = baseline_results[name]["accuracy"]
    t_stat, p_val = stats.ttest_1samp(proposed_accs, baseline_acc)
    ttest_rows.append({
        "Comparison": f"Proposed vs {name}",
        "t-test": round(float(t_stat), 4),
        "P-value": f"{p_val:.2e}",
        "Significant (p<0.05)": "Yes" if p_val < 0.05 else "No"
    })

ttest_df = pd.DataFrame(ttest_rows)
ttest_df


## 16. Noise-Robustness Analysis

Evaluates the trained FSMW-DDS model on test signals with additive white Gaussian noise injected at several SNR levels (before VMD + MRWA re-extraction), reproducing the paper's robustness table.

In [ ]:
def add_awgn(signal: np.ndarray, snr_db: float, rng) -> np.ndarray:
    sig_power = np.mean(signal ** 2)
    snr_linear = 10 ** (snr_db / 10)
    noise_power = sig_power / snr_linear
    noise = rng.normal(0, np.sqrt(noise_power), size=signal.shape)
    return (signal + noise).astype(np.float32)


def evaluate_noise_robustness(model, X_raw_test, y_test, cfg, device, snr_levels=(20, 10, 5)):
    rng = np.random.default_rng(SEED)
    results = {}

    def eval_at(feats_norm):
        ds_ = PCGFeatureDataset(feats_norm, y_test)
        return evaluate_model(model, ds_, device, cfg)

    # Clean
    clean_feats = preprocess_corpus(X_raw_test, cfg, use_vmd=True, use_mrwa=True, max_frames=MAX_FRAMES)
    clean_feats_norm = (clean_feats - mu) / sigma
    results["Clean (No Noise)"] = eval_at(clean_feats_norm)

    for snr in snr_levels:
        noisy_X = np.stack([add_awgn(s, snr, rng) for s in X_raw_test])
        noisy_feats = preprocess_corpus(noisy_X, cfg, use_vmd=True, use_mrwa=True, max_frames=MAX_FRAMES)
        noisy_feats_norm = (noisy_feats - mu) / sigma
        results[f"{snr} dB"] = eval_at(noisy_feats_norm)
        print(f"SNR={snr:>3} dB -> acc={results[f'{snr} dB']['accuracy']*100:.2f}%  "
              f"f1={results[f'{snr} dB']['f1']*100:.2f}%")

    return results


noise_results = evaluate_noise_robustness(
    full_model, X_raw[X_test_idx], LABELS[X_test_idx], CFG, DEVICE, snr_levels=(20, 10, 5))

noise_df = pd.DataFrame(noise_results).T[["accuracy", "precision", "recall", "f1", "mcc"]]
noise_df[["accuracy", "precision", "recall", "f1"]] = (noise_df[["accuracy", "precision", "recall", "f1"]] * 100).round(2)
noise_df["mcc"] = noise_df["mcc"].round(4)
noise_df


## 17. Federated-Learning Round-by-Round Performance

In [ ]:
round_checkpoints = [5, 10, 20, 30, 40, 50]
fed_round_table = []
for r in round_checkpoints:
    r_clamped = min(r, len(full_history["accuracy"])) - 1
    fed_round_table.append({
        "Round": r,
        "Accuracy (%)": round(full_history["accuracy"][r_clamped] * 100, 2),
        "Precision (%)": round(full_history["precision"][r_clamped] * 100, 2),
    })
pd.DataFrame(fed_round_table).set_index("Round")


## 18. Computational Efficiency & Deployment Cost Comparison

In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def measure_inference_latency(model: nn.Module, sample_input: torch.Tensor, device, n_reps: int = 50) -> float:
    model.eval().to(device)
    sample_input = sample_input.to(device)
    # warm-up
    with torch.no_grad():
        for _ in range(5):
            model(sample_input)
    if device.type == "cuda":
        torch.cuda.synchronize()
    import time
    start = time.time()
    with torch.no_grad():
        for _ in range(n_reps):
            model(sample_input)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = (time.time() - start) / n_reps
    return elapsed * 1000.0  # ms


sample_batch = torch.zeros(1, MAX_FRAMES, FEAT_DIM)

models_for_cost = {
    "1D CNN": Simple1DCNN(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "CNN+LSTM": CNNLSTM(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "ResNet-1D": ResNet1D(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "ViT-style": ViTStyle(FEAT_DIM, MAX_FRAMES, CFG.n_classes),
    "FSMW-DDS (Proposed)": full_model,
}

cost_rows = []
for name, m in models_for_cost.items():
    n_params = count_params(m)
    latency = measure_inference_latency(m, sample_batch, DEVICE)
    model_size_mb = n_params * 4 / (1024 ** 2)  # float32
    cost_rows.append({
        "Method": name,
        "Parameters (M)": round(n_params / 1e6, 3),
        "Model Size (MB)": round(model_size_mb, 2),
        "Inference Latency (ms)": round(latency, 3),
    })

cost_df = pd.DataFrame(cost_rows).set_index("Method")
cost_df


## 19. Confusion Matrix & Class-wise Report for the Final Model

In [ ]:
@torch.no_grad()
def get_predictions(model, dataset, device, cfg):
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False)
    all_y, all_pred = [], []
    model.eval()
    for xb, yb in loader:
        xb = xb.to(device)
        _, _, pred, _ = model(xb)
        all_y.append(yb.numpy()); all_pred.append(pred.cpu().numpy())
    return np.concatenate(all_y), np.concatenate(all_pred)

y_true, y_pred = get_predictions(full_model, test_ds, DEVICE, CFG)
cm = confusion_matrix(y_true, y_pred, labels=list(range(CFG.n_classes)))

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(CFG.n_classes), CFG.class_names, rotation=45, ha="right")
plt.yticks(range(CFG.n_classes), CFG.class_names)
for i in range(CFG.n_classes):
    for j in range(CFG.n_classes):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                  color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix -- FSMW-DDS (Test Set)")
plt.tight_layout(); plt.show()


## 20. Save the Trained Model

In [ ]:
os.makedirs("/mnt/user-data/outputs", exist_ok=True)
save_path = "/mnt/user-data/outputs/fsmw_dds_model.pt"
torch.save({
    "model_state_dict": full_model.state_dict(),
    "config": CFG.__dict__,
    "feature_mean": mu,
    "feature_std": sigma,
}, save_path)
print("Saved trained FSMW-DDS model to:", save_path)


## 21. Summary

This notebook implements every module described in the paper end-to-end and reproducible:

| Module | Section | Equations |
|---|---|---|
| VMD signal enhancement | §3 | (2)–(7) |
| MRWA hierarchical features | §4 | (8)–(13) |
| CDPM synthetic data generation | §6 | (14)–(23) |
| MSAWT transformer | §7 | (24)–(37) |
| FedProx federated optimization | §9 | (38)–(47) |
| ACACC confidence-aware classifier | §8 | (48)–(57) |

**To reproduce paper-scale numbers:** point `CFG.data_dir` at a local copy of the PhysioNet/CinC Challenge 2016 dataset, increase `comm_rounds` to 50, `diffusion_epochs` to 100+, and run on a GPU. The synthetic-data fallback used above is only meant to make every cell runnable without external downloads — absolute metric values on synthetic data are **not** directly comparable to the paper's reported PhysioNet results.
